# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/boddulavinaykumar6-stack/Flyrank-ML-Internship/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

*Name your lane — or say 'freestyle' and describe your own question. One short paragraph: why this one?*


**Selected Lane:** Refresh / Content Opportunity Scoring (Lane 2)

I selected the Refresh / Content Opportunity Scoring lane because it focuses on identifying which content pages should be reviewed first based on observable search and engagement signals. Websites often contain many pages, making manual review difficult and time-consuming. This project aims to support decision-making by ranking pages that appear to have the greatest opportunity for review using measurable data rather than assumptions.

In [7]:
!git clone https://github.com/boddulavinaykumar6-stack/Flyrank-ML-Internship.git

Cloning into 'Flyrank-ML-Internship'...
remote: Enumerating objects: 127, done.
remote: Counting objects: 100% (127/127), done.
remote: Compressing objects: 100% (84/84), done.
remote: Total 127 (delta 41), reused 93 (delta 27), pack-reused 0 (from 0)
Receiving objects: 100% (127/127), 1.85 MiB | 13.07 MiB/s, done.
Resolving deltas: 100% (41/41), done.


In [8]:
%cd /content/Flyrank-ML-Internship

/content/Flyrank-ML-Internship


In [9]:
!ls data/raw

content_refresh_anonymized.csv


# 2. The Question

### Research Question

Which content pages should be prioritized for review based on historical search performance and engagement metrics?

### What decision does this improve?

It helps content editors decide which pages should be refreshed first.

### Who acts on the output?

Content editors and SEO specialists use the ranked list to decide which pages to review.

### What does a wrong answer cost?

- False positives waste editor time by recommending pages that do not need updates.
- False negatives miss pages that should have been refreshed, potentially reducing search traffic or engagement.

### Why does machine learning help?

A simple rule based on one metric cannot capture the interaction among many search and engagement signals. A ranking model can combine multiple signals to prioritize pages more effectively.

### Task Type

Ranking / Scoring

### Desired Output

A refresh priority score for every content page.

### Success Metric

Precision@K

### One-Paragraph Frame

For content editors deciding which pages to refresh first, we will build a ranking model using historical search performance and engagement data. The model will assign each page a refresh priority score. Performance will eventually be evaluated using Precision@K. Incorrect recommendations either waste editor time or miss valuable refresh opportunities. A simple rule is insufficient because refresh priority depends on multiple interacting signals. The model will provide decision support based on observed historical patterns rather than causal claims.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*

# 3. Quick Look at the Data

The starter dataset contains **30,000 rows** and **44 columns**, where each row represents one pseudonymized content item.

The dataset contains search performance, engagement, and content-related metrics collected over a trailing 90-day window.

Some variables contain missing values, which will need to be handled carefully during preprocessing. The project documentation notes that missingness depends on content type, so missing values should not automatically be replaced with zero.

The documentation also notes that percentage columns (such as CTR) are stored as percentage values rather than fractions, and that `avg_position = 0` represents missing data rather than a true ranking.

In [16]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

df.shape

(30000, 44)

In [19]:
print(df.shape)
print(df.columns.tolist())
print(df.isnull().sum())

(30000, 44)
['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']
content_id                    0
client_id                     0
search_volume              2468
competition                2468
competition_level          2610
cpc                        2468
content_type             

In [17]:
df.head()

,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


In [18]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30000 entries, 0 to 29999
Data columns (total 44 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   content_id              30000 non-null  object 
 1   client_id               30000 non-null  object 
 2   search_volume           27532 non-null  float64
 3   competition             27532 non-null  float64
 4   competition_level       27390 non-null  object 
 5   cpc                     27532 non-null  float64
 6   content_type            30000 non-null  object 
 7   main_intent             27626 non-null  object 
 8   word_count              22301 non-null  float64
 9   char_count              22301 non-null  float64
 10  provider_used           8562 non-null   object 
 11  model_used              24267 non-null  object 
 12  impressions_90d         30000 non-null  int64  
 13  clicks_90d              30000 non-null  int64  
 14  pageviews_90d           30000 non-null

# 4. Careful Words

This project is intended to support decision-making rather than establish causal relationships.

The analysis will identify historical patterns associated with pages that may benefit from review. Any conclusions describe observed associations and should not be interpreted as evidence that any feature directly causes changes in search performance.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


# 5. Self-Check

- ✅ Selected a project lane.
- ✅ Defined the decision the model supports.
- ✅ Identified who will use the output.
- ✅ Explained the cost of incorrect recommendations.
- ✅ Selected the ML task type (Ranking / Scoring).
- ✅ Examined the dataset.
- ✅ Used careful observational language.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.